In [125]:
#creating customer dataset
customer_data = df.groupby(
    'Customer ID'
).agg({

    'InvoiceDate':'max',

    'Invoice':'nunique',

    'TotalAmount':'sum'
})
print(customer_data.head())

                    InvoiceDate  Invoice  TotalAmount
Customer ID                                          
12346.0     2011-01-18 10:01:00        1     77183.60
12347.0     2011-12-07 15:52:00        7      4310.00
12348.0     2011-09-25 13:13:00        4      1797.24
12349.0     2011-11-21 09:51:00        1      1757.55
12350.0     2011-02-02 16:01:00        1       334.40


In [127]:
#renaming the columns
customer_data.columns = [

    'LastPurchaseDate',

    'TotalOrders',

    'TotalRevenue'
]
print(customer_data.head())

               LastPurchaseDate  TotalOrders  TotalRevenue
Customer ID                                               
12346.0     2011-01-18 10:01:00            1      77183.60
12347.0     2011-12-07 15:52:00            7       4310.00
12348.0     2011-09-25 13:13:00            4       1797.24
12349.0     2011-11-21 09:51:00            1       1757.55
12350.0     2011-02-02 16:01:00            1        334.40


In [1]:
#Calculate Recency
snapshot_date = (
    df['InvoiceDate'].max()
    + pd.Timedelta(days=1)
)
customer_data['AverageOrderValue'] = (
    customer_data['TotalRevenue'] /
    customer_data['TotalOrders']
).dt.days

NameError: name 'df' is not defined

In [129]:
#Create Churn Label
#Rule:
#Days Since Last Purchase >= 90

#Churn = 1 otherwise Churn = 0

customer_data['Churn'] = np.where(

    customer_data[
        'DaysSinceLastPurchase'
    ] >= 90,

    1,

    0
)
customer_data[
    'Churn'
].value_counts()

,count
Churn,
0,2880
1,1458


In [130]:
#Create features
X = customer_data[[

    'DaysSinceLastPurchase',

    'TotalOrders',

    'TotalRevenue'

]]
y = customer_data['Churn']

In [131]:
#Train, Test and Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42
)

In [132]:
!pip install xgboost

In [134]:
#Train XGBoost
from xgboost import XGBClassifier

model = XGBClassifier(

    random_state=42
)

model.fit(

    X_train,

    y_train
)

#Prediction
predictions = model.predict(
    X_test
)


In [135]:
#Evaluation
from sklearn.metrics import (

    accuracy_score,

    classification_report,

    confusion_matrix
)

print(

    accuracy_score(
        y_test,
        predictions
    )
)

print(

    classification_report(
        y_test,
        predictions
    )
)

1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       560
           1       1.00      1.00      1.00       308

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868



In [136]:
#Saving the Results
customer_data.to_csv(

    "customer_churn_data.csv"
)

In [137]:
print(customer_data.shape)

print(
customer_data['Churn']
.value_counts()
)

print(
accuracy_score(
y_test,
predictions
)
)

print(
classification_report(
y_test,
predictions
)
)

(4338, 5)
Churn
0    2880
1    1458
Name: count, dtype: int64
1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       560
           1       1.00      1.00      1.00       308

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868

